In [ ]:
# ============================================================
# IAIAS-QP FULL PIPELINE IMPLEMENTATION (ONE FILE)
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestRegressor

import shap

import tensorflow as tf
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model

# ============================================================
# 1. LOAD DATA
# ============================================================

df = pd.read_csv("MiningProcess_Flotation_Plant_Database.csv")

# Drop date column
df = df.drop(columns=['date'], errors='ignore')

# ============================================================
# 2. PREPROCESSING
# ============================================================

# Handle missing
df = df.dropna()

# Target
target = "% Silica Concentrate"

X = df.drop(columns=[target])
y = df[target]

# Normalization
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ============================================================
# 3. FEATURE SELECTION (RFE + SBFS Approx)
# ============================================================

estimator = RandomForestRegressor(n_estimators=100, random_state=42)

rfe = RFE(estimator, n_features_to_select=15)
X_rfe = rfe.fit_transform(X_scaled, y)

selected_features = X.columns[rfe.support_]

print("Selected Features:", selected_features)

# ============================================================
# 4. SEQUENCE CREATION (IMPORTANT FOR TCN + BiGRU)
# ============================================================

def create_sequences(X, y, time_steps=10):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:i+time_steps])
        ys.append(y.iloc[i+time_steps])
    return np.array(Xs), np.array(ys)

X_seq, y_seq = create_sequences(pd.DataFrame(X_rfe), y)

X_train, X_test, y_train, y_test = train_test_split(
    X_seq, y_seq, test_size=0.2, random_state=42
)

# ============================================================
# 5. MODEL: TCN + BiGRU + MHA
# ============================================================

def build_model(input_shape):

    inputs = Input(shape=input_shape)

    # -------- TCN BLOCK --------
    x = Conv1D(64, kernel_size=3, padding='causal', dilation_rate=1, activation='relu')(inputs)
    x = Conv1D(64, kernel_size=3, padding='causal', dilation_rate=2, activation='relu')(x)
    x = Dropout(0.2)(x)

    # -------- BiGRU --------
    x = Bidirectional(GRU(64, return_sequences=True))(x)

    # -------- Multi-Head Attention --------
    attn_output = MultiHeadAttention(num_heads=4, key_dim=32)(x, x)
    x = Add()([x, attn_output])
    x = LayerNormalization()(x)

    x = GlobalAveragePooling1D()(x)

    # -------- Dense --------
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.2)(x)

    outputs = Dense(1)(x)

    model = Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )

    return model

model = build_model(X_train.shape[1:])
model.summary()

# ============================================================
# 6. TRAINING
# ============================================================

history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=64,
    validation_data=(X_test, y_test),
    verbose=1
)

# ============================================================
# 7. EVALUATION
# ============================================================

y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MSE: {mse}")
print(f"RMSE: {rmse}")
print(f"MAE: {mae}")
print(f"R2: {r2}")

# ============================================================
# 8. PLOTS (ALL YOUR PAPER FIGURES)
# ============================================================

# Correlation Heatmap
plt.figure(figsize=(12,8))
sns.heatmap(df.corr(), cmap='coolwarm')
plt.title("Correlation Matrix")
plt.show()

# Histogram + KDE
sns.histplot(df[target], kde=True)
plt.title("Silica Distribution")
plt.show()

# Scatter
sns.scatterplot(x="% Iron Feed", y="% Iron Concentrate", data=df)
plt.title("Iron Feed vs Concentrate")
plt.show()

# Training Curve
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.legend(['Train', 'Test'])
plt.title("Loss Curve")
plt.show()

plt.figure(figsize=(8,5))

plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Test Loss')

plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")

plt.legend()
plt.show()

plt.figure(figsize=(8,5))
plt.plot(history.history['mae'], label='Train MAE')
plt.plot(history.history['val_mae'], label='Val MAE')
plt.title('MAE Curve')
plt.xlabel('Epochs')
plt.ylabel('MAE')
plt.legend()
plt.show()

plt.figure(figsize=(8,5))
plt.plot(history.history['rmse'], label='Train RMSE')
plt.plot(history.history['val_rmse'], label='Val RMSE')
plt.title('RMSE Curve')
plt.xlabel('Epochs')
plt.ylabel('RMSE')
plt.legend()
plt.show()

plt.figure(figsize=(8,5))
plt.plot(history.history['mape'], label='Train MAPE')
plt.plot(history.history['val_mape'], label='Val MAPE')
plt.title('MAPE Curve')
plt.xlabel('Epochs')
plt.ylabel('MAPE')
plt.legend()
plt.show()

plt.figure(figsize=(8,5))
plt.plot(history.history['r2_score'], label='Train R²')
plt.plot(history.history['val_r2_score'], label='Val R²')
plt.title('R² Curve')
plt.xlabel('Epochs')
plt.ylabel('R²')
plt.legend()
plt.show()
# 6. TRAIN vs TEST PERFORMANCE PLOT
# ------------------------------
train_preds = dl_model.predict(X_train).flatten()

train_mse = mean_squared_error(y_train, train_preds)
test_mse = mean_squared_error(y_test, preds_dl)

train_rmse = np.sqrt(train_mse)
test_rmse = np.sqrt(test_mse)

train_mae = mean_absolute_error(y_train, train_preds)
test_mae = mean_absolute_error(y_test, preds_dl)

train_r2 = r2_score(y_train, train_preds)
test_r2 = r2_score(y_test, preds_dl)

metrics = ["MSE", "RMSE", "MAE", "R2"]
train_values = [train_mse, train_rmse, train_mae, train_r2]
test_values = [test_mse, test_rmse, test_mae, test_r2]

x = np.arange(len(metrics))

plt.figure(figsize=(8,5))
plt.bar(x - 0.2, train_values, width=0.4, label="Train")
plt.bar(x + 0.2, test_values, width=0.4, label="Test")
plt.xticks(x, metrics)
plt.title("Training vs Testing Performance")
plt.legend()
plt.show()


# 8. MODEL COMPARISON (MSE)
# ------------------------------
model_names = list(metrics_df.index)
mse_values = metrics_df["MSE"]

plt.figure(figsize=(8,5))
plt.bar(model_names, mse_values)
plt.title("Model Comparison (MSE)")
plt.ylabel("Error")
plt.xticks(rotation=30)
plt.show()

# ============================================================
# 9. SHAP EXPLAINABILITY
# ============================================================

# Use smaller subset (SHAP is heavy)
X_sample = X_train[:100]

explainer = shap.DeepExplainer(model, X_sample)
shap_values = explainer.shap_values(X_sample)

shap.summary_plot(shap_values[0], X_sample)

# ============================================================
# END
# ============================================================